# 2. A strategy, end to end

Write a strategy, run it in the simulator, and read what happened: PnL and its parts, transaction costs,
markouts, the queue and risk, then the tearsheet. The strategy here is a variation of the passive quoter that
stops quoting the side the order book leans against, compared with the C++ `passive_quoter` on the same data.

It runs on the committed one-minute slice; set `CONFIG` for the recorded day (commented below).

In [ ]:
from pathlib import Path

import numpy as np
import polars as pl
from IPython.display import display

import blocksprint_sim as bs
from research.analytics import report

# The repository root, wherever the notebook is opened from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "research" / "analytics").is_dir())

from research.analytics import markouts, pnl, queue, risk, tca
from research.sim.results import Results
from research.strategies import Context, Strategy

OUT = ROOT / "results" / "notebooks"
CONFIG = {
    "dataset": str(ROOT / "tests/sim/data/btcusdt_2026-11-05_0200_1m.bse"),
    "latency_preset": "binance_spot_tokyo_vps",
    "queue_model": {"model": "cancel_from_back"},
    "fees": {"maker_rate": "0.001", "taker_rate": "0.001"},
    "strategy": {"name": "null"},  # the C++ side runs none; the Python object passed to Results.run trades
    "output": {"dir": str(OUT / "walkthrough"), "sample_ms": 1000},
}
# CONFIG.update({"dataset": str(ROOT / "data/packed/binance_spot_btc_eth_1110"), "symbol": "BTCUSDT",
#                "window": {"from": "2026-11-10T02:00:00Z", "to": "2026-11-10T03:00:00Z", "warmup_s": 60}})

## The strategy

Every `refresh_ms` it reads the book it sees (feed-delayed), and quotes `size` at the best bid and the best
ask, post-only, like the passive quoter, except:

- when the top-level imbalance is above `skip`, the book leans to the bid and the price tends to rise next:
  a resting ask would be picked off, so it isn't quoted (and the bid when the imbalance is below `-skip`);
- a side whose next fill would pass `max_position` isn't quoted.

Units are the simulator's: prices and quantities are integers in 1e-8 (`bs.to_fixed("0.001") == 100_000`).

In [ ]:
class ImbalanceAwareQuoter(Strategy):
    callback_mode = "timer_only"

    def __init__(self, size="0.001", refresh_ms=250, skip=0.5, max_position="0.005"):
        self.size, self.max_position = bs.to_fixed(size), bs.to_fixed(max_position)
        self.refresh_ns, self.skip = int(refresh_ms * 1_000_000), skip

    def on_start(self, ctx: Context) -> None:
        ctx.set_timer(self.refresh_ns)

    def on_timer(self, ctx: Context, timer_id: int) -> None:
        self.requote(ctx)
        ctx.set_timer(self.refresh_ns)

    def requote(self, ctx: Context) -> None:
        if not ctx.can_trade():
            return
        book, position = ctx.book(), ctx.account().position
        bid = ask = None
        if book.valid() and (b := book.best_bid()) and (a := book.best_ask()):
            imbalance = (b.qty - a.qty) / (b.qty + a.qty)
            if imbalance > -self.skip and position + self.size <= self.max_position:
                bid = b.price
            if imbalance < self.skip and position - self.size >= -self.max_position:
                ask = a.price
        orders = ctx.open_orders()
        for side, price in ((bs.Side.BUY, bid), (bs.Side.SELL, ask)):
            mine = [o for o in orders.values() if o.side == side]
            if any(o.status != bs.OrderStatus.NEW and o.status != bs.OrderStatus.PARTIALLY_FILLED for o in mine):
                continue  # a request is in flight: wait for its answer
            if price is None:
                for o in mine:
                    ctx.cancel(o.id)
            elif not mine:
                ctx.send(side, bs.OrderType.LIMIT_MAKER, price, self.size)
            elif mine[0].price != price:
                ctx.replace(mine[0].id, price, self.size)

## Run it, and the baseline

`Results.run` runs one simulation in this process and keeps every table in memory. The baseline is the C++
`passive_quoter` with the same size, refresh and limit.

In [ ]:
mine = Results.run(CONFIG, ImbalanceAwareQuoter())
base = Results.run({**CONFIG, "strategy": {"name": "passive_quoter", "params": {
    "size": "0.001", "refresh_ms": 250, "max_position": "0.005"}},
    "output": {**CONFIG["output"], "dir": str(OUT / "baseline")}})
runs = {"imbalance-aware": mine, "passive quoter": base}
pl.concat([pnl.decompose(r).with_columns(pl.lit(name).alias("strategy")) for name, r in runs.items()]).select(
    "strategy", "pnl", "spread_capture", "inventory", "fees", "fills", "edge_bps", "position")

## Costs, markouts, queue, risk

`tca.summary`: fill ratio, time to fill, the effective and realized spread and the price impact at 5 s (costs
positive when paid). `markouts.markout_curve`: the fills' value at the mid some time later (positive is good).
If the imbalance predicts the next move, skipping the side the book leans against lifts the curve above the
baseline's; one minute of data has too few fills to tell, so try the recorded hour.

In [ ]:
pl.concat([tca.summary(r).with_columns(pl.lit(name).alias("strategy")) for name, r in runs.items()]).select(
    "strategy", "placements", "fill_ratio", "ttf_p50_ms", "effective_spread_bps", "realized_spread_bps",
    "price_impact_bps", "cancel_fill_ratio")

In [ ]:
curves = {name: markouts.markout_curve(r, by=()) for name, r in runs.items()}
labels = curves["passive quoter"]["horizon"].to_list()  # 0ms ... 60s (horizons past the data's end are left out)
series = [(name, [dict(zip(c["horizon"], c["mean_bps"])).get(h, np.nan) for h in labels]) for name, c in curves.items()]
display(report.frame(report.card("Markout (bps, all fills)", report.line_chart(
    list(range(len(labels))), series, x_labels=labels, x_ticks=list(enumerate(labels)), y_title="bps", digits=3,
    table_x="horizon")), 440))

In [ ]:
pl.concat([queue.summary(r).with_columns(pl.lit(name).alias("strategy")) for name, r in runs.items()]).select(
    "strategy", "placements", "median_ahead", "median_time_in_queue_ms", "fill_rate", "fifo_reachable")

In [ ]:
every = report.auto_every(mine.meta["window"]["end_ns"] - mine.meta["window"]["trading_ns"])  # >= 30 buckets
pl.concat([risk.summary(r, every).with_columns(pl.lit(name).alias("strategy")) for name, r in runs.items()]).select(
    "strategy", "pnl", "max_drawdown", "sharpe", "inv_mean_abs", "time_at_position_limit", "holding_time_s")

## The tearsheet

The same page `python -m research.run` writes for every run of an experiment (`report.write_tearsheet(run,
path)` saves it). A strategy defined in a notebook runs in this process only: to sweep it with
`python -m research.run`, which runs in worker processes, put it in a module (`research/strategies/`).

In [ ]:
page = report.tearsheet(mine, title="Imbalance-aware quoter")
display(report.frame(page, 1600, title="tearsheet"))